# Advanced Statistics for Business and Energy Decisions

**Course alignment:** Advanced Statistical Methods  
**Type:** Original supplementary practical notebook  
**Scenario:** Simulated electricity service and customer operations

This notebook puts several statistical methods from the course into one applied workflow: descriptive statistics, confidence intervals, Welch's t-test, ANOVA, correlation, categorical association, regression and Bayesian updating. The dataset is synthetic and must not be treated as real utility or customer evidence.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

rng = np.random.default_rng(2026)
n = 900
tariff = rng.choice(["Standard","Time-of-use","Incentive"], n, p=[.4,.35,.25])
region = rng.choice(["Urban","Peri-urban","Rural"], n, p=[.45,.30,.25])
temperature = rng.normal(29, 4, n)
household_size = rng.integers(1, 8, n)
tariff_effect = pd.Series(tariff).map({"Standard":0,"Time-of-use":-7,"Incentive":-3}).to_numpy()
region_effect = pd.Series(region).map({"Urban":15,"Peri-urban":5,"Rural":-8}).to_numpy()
kwh = np.maximum(20, 65 + 2.2*(temperature-25) + 5*household_size + tariff_effect + region_effect + rng.normal(0,12,n))
monthly_bill = 12 + 8.5*kwh + rng.normal(0,55,n)
complaints = rng.poisson(np.exp(-1.3 + .018*np.maximum(temperature-25,0) + .012*household_size))
df = pd.DataFrame({
    "tariff":tariff,"region":region,"temperature_c":temperature,
    "household_size":household_size,"monthly_kwh":kwh,
    "monthly_bill":monthly_bill,"complaints":complaints
})
display(df.head())
display(df.describe(include="all").T)

## 1. Descriptive statistics and confidence interval

A confidence interval communicates sampling uncertainty around an estimate. It does not describe the range containing 95% of individual household observations.

In [ ]:
summary = df.groupby("tariff")["monthly_kwh"].agg(["count","mean","median","std"])
display(summary.round(2))

group = df.loc[df["tariff"]=="Standard","monthly_kwh"]
mean = group.mean()
se = stats.sem(group)
ci = stats.t.interval(.95, df=len(group)-1, loc=mean, scale=se)
print(f"Standard tariff mean: {mean:.2f} kWh")
print(f"95% t-based CI for mean: [{ci[0]:.2f}, {ci[1]:.2f}] kWh")

## 2. Two-group comparison: Welch's t-test

The test compares the mean monthly consumption of Standard and Time-of-use groups. Welch's test does not require equal population variances. A causal interpretation would still require a credible assignment design; this observational synthetic example only compares groups.

In [ ]:
standard = df.loc[df.tariff=="Standard","monthly_kwh"]
tou = df.loc[df.tariff=="Time-of-use","monthly_kwh"]
levene = stats.levene(standard, tou, center="median")
welch = stats.ttest_ind(standard, tou, equal_var=False)
print("Levene p-value:", levene.pvalue)
print("Welch t statistic:", welch.statistic)
print("Welch p-value:", welch.pvalue)
print("Mean difference (Standard - Time-of-use):", standard.mean()-tou.mean())

## 3. Three-group comparison: one-way ANOVA

ANOVA tests whether all group means are equal. A significant omnibus result says at least one group mean differs; it does not identify which pair differs. Post-hoc comparisons require an appropriate multiple-comparison correction.

In [ ]:
groups = [part["monthly_kwh"].to_numpy() for _,part in df.groupby("tariff")]
anova = stats.f_oneway(*groups)
grand_mean = df["monthly_kwh"].mean()
ss_between = sum(len(g)*(g.mean()-grand_mean)**2 for g in groups)
ss_total = ((df["monthly_kwh"]-grand_mean)**2).sum()
eta_squared = ss_between/ss_total
print("ANOVA F:", anova.statistic)
print("ANOVA p-value:", anova.pvalue)
print("Eta-squared:", eta_squared)
display(df.groupby("tariff")["monthly_kwh"].mean().sort_values().to_frame("mean_kwh"))

## 4. Correlation: choose the coefficient to match the question

Pearson measures linear association; Spearman and Kendall use ranks and can be useful for monotonic relationships or ordinal variables. Correlation does not establish causation.

In [ ]:
for name, result in {
    "Pearson":stats.pearsonr(df["temperature_c"],df["monthly_kwh"]),
    "Spearman":stats.spearmanr(df["temperature_c"],df["monthly_kwh"]),
    "Kendall":stats.kendalltau(df["temperature_c"],df["monthly_kwh"])
}.items():
    print(name, "statistic=", round(result.statistic,3), "p=", f"{result.pvalue:.4g}")

fig, ax = plt.subplots(figsize=(7,4))
ax.scatter(df["temperature_c"],df["monthly_kwh"],alpha=.25,s=12)
ax.set(title="Temperature and monthly electricity use (simulated)",xlabel="Temperature (°C)",ylabel="Monthly use (kWh)")
plt.tight_layout()
plt.show()

## 5. Categorical association: chi-square and Cramér's V

In [ ]:
contingency = pd.crosstab(df["region"],df["tariff"])
chi2, p, dof, expected = stats.chi2_contingency(contingency)
n_obs = contingency.to_numpy().sum()
r,c = contingency.shape
cramers_v = np.sqrt(chi2/(n_obs*min(r-1,c-1)))
display(contingency)
print("Chi-square:", chi2, "df:", dof, "p-value:", p)
print("Cramer's V:", cramers_v)
print("Minimum expected count:", expected.min())

## 6. Multiple linear regression

The model estimates monthly consumption using temperature, household size, tariff and region. Coefficients are conditional associations holding the included predictors constant. Check residual patterns, multicollinearity and omitted-variable risks before using the model for decisions.

In [ ]:
ols = smf.ols(
    "monthly_kwh ~ temperature_c + household_size + C(tariff) + C(region)",
    data=df
).fit()
print(ols.summary())

## 7. Bayesian updating: a rare-event screening example

Use Bayes' rule to calculate the probability of a true event after a positive screen. The example uses hypothetical prevalence, sensitivity and specificity; these are not clinical estimates.

In [ ]:
prevalence = 0.02
sensitivity = 0.90
specificity = 0.95
p_positive = sensitivity*prevalence + (1-specificity)*(1-prevalence)
posterior = sensitivity*prevalence/p_positive
print("P(positive):", round(p_positive,4))
print("P(event | positive):", round(posterior,4))
print("Posterior probability:", f"{posterior:.1%}")

## Interpretation tasks

1. Distinguish statistical significance from practical significance in the tariff comparison.
2. Explain why the ANOVA result needs post-hoc testing before naming a specific tariff pair.
3. Compare Pearson and Spearman results. Which best describes the relationship?
4. Interpret Cramér's V as an effect-size measure, not a p-value.
5. Explain why a low base rate can produce a modest posterior probability even with a good screening test.
6. Identify at least three limitations that would matter when repeating this analysis with actual meter data.

**Data caveat:** Every observation in this notebook is simulated.